# Preprocessing: Tafel Slopes & Overpotentials from Raw CV Data

Reads raw cyclic voltammetry (CV) files from the HT-SDC instrument, extracts the
5th forward scan, applies iR correction, converts to overpotential η, and writes
two output CSVs:

- `composition_Tafel.csv` — quaternary composition + Tafel slope (V/decade)
- `composition_eta.csv`   — quaternary composition + η at 10 mA/cm²

Differential Tafel (dJ/dV) CSVs and per-sample plots are saved to `dJdV/`.

In [1]:
import zipfile
import os

import numpy as np
import pandas as pd
import matplotlib

matplotlib.use('Agg')  # Headless backend: figures are saved, never displayed.
import matplotlib.pyplot as plt

from oer_utils import (
    load_catalyst_summary,
    list_data_files,
    match_composition,
    composition_values,
)

# ── Physical constants ───────────────────────────────────────
Vo = 1.23  # OER equilibrium potential vs RHE (V) at 25 °C.
AREA = 0.165  # Electrode area (cm²)

# ── OER-branch extraction: vertical line test ────────────────
# Trace the smoothed scan left to right in η vs log₁₀(J). η is
# monotone in scan order, so the curve's slope passes through infinity
# exactly where J reverses direction. For the S-shaped pre-OER oxide
# fold, the first vertical tangent (J rising → falling) is the
# lower-right fold — the wrong vertical line — and the second
# (falling → rising) is the upper-left fold. The OER branch is
# everything after the second, and the Tafel region begins almost
# immediately afterward. A scan with no second vertical tangent is
# single-valued and is used whole; its Tafel region is then the last
# plateau before the terminal conditions become unsatisfactory at the
# right edge of the graph. Noise robustness comes from persistence
# only: a direction reversal must last TURN_MIN_RUN points to count.
TURN_MIN_RUN = 9  # Persistence (scan points); the oxide fold spans
#                   ~100 points, instrument wiggles far fewer.
ETA_MIN = 0.05  # Tafel validity floor (V): below this the back reaction
#                 and residual oxide currents contaminate the curve.
SNR_MIN = 2.0  # Background-corrected J must exceed this multiple of the
#                background to count as OER signal.

# ── Sliding-window Tafel detection on the extracted branch ───
WINDOW_DECADES = 0.35  # Width of each local-slope window (decades of J).
DENSITY_MIN = 20.0  # Min points per decade inside a window (fit stability).
PLATEAU_TOL = 0.30  # Local slopes within (1+tol)×plateau join the fit region;
#                     the same tolerance defines the transport upturn.
LOCAL_R2_THRESHOLDS = (0.90, 0.80)  # Local-linearity gates, strictest first.
SLOPE_MAX = 0.25  # V/decade plausibility ceiling (Butler–Volmer, α ≳ 0.25).
MIN_FIT_DECADES = 0.3  # Reject fits spanning less than this; slope → NaN.
#                        Short spans are honest on scan data with a
#                        detected transport bend; the span is reported.

# ── Paths ────────────────────────────────────────────────────
ZIP_PATH = 'data/CV.zip'
CV_DIR = 'data/CV/'
DJDV_DIR = 'data/dJdV/'
IMAGE_DIR = 'images/'
TAFEL_DIR = 'images/Tafel/'  # Journal-style plots: fit region only.
TAFEL_DIAG_DIR = 'images/TafelDiagnostics/'  # Full curve + detected region.

# Unzip the CV archive once; it extracts a CV/ folder into data/.
if not os.path.exists(CV_DIR):
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall('data')


In [2]:
def find_second_turning_point(Js):
    """Scan index of the second vertical tangent (the upper-left fold).

    Vertical line test, applied to the curve as plotted: in η–log₁₀(J)
    coordinates η is monotone in scan order, so the curve's slope
    passes through infinity exactly where J reverses direction, and
    the plot only contains points with J > 0. Tracing the smoothed
    scan left to right, the first vertical tangent (J rising →
    falling) is the lower-right fold of the S — the wrong vertical
    line — and the second (falling → rising) is the upper-left fold,
    where the OER branch begins.

    Robustness comes from persistence alone, with no magnitude
    thresholds of any kind:
    a direction run must last at least ``TURN_MIN_RUN`` points to
    count, shorter wiggles are absorbed into the run before them, and
    detection is restricted to the trailing stretch of positive
    current (points that appear on the log plot). The second turning
    point is located as the start of the rising run that carries the
    curve to its terminal maximum — on an S-shaped scan this is
    identical to counting two reversals from the left, but it cannot
    be faked by noise in the low-current region. Returns ``None`` when
    the curve never folds (single-valued): no rising run contains the
    maximum, or that run extends back to the start of the trace.
    """
    Js = np.asarray(Js, dtype=float)

    # The plotted curve: trailing contiguous stretch of J > 0.
    nonpos = np.where(Js <= 0)[0]
    i0 = nonpos[-1] + 1 if len(nonpos) else 0
    seg = Js[i0:]
    if len(seg) < 3 * TURN_MIN_RUN:
        return None

    d = np.diff(seg)
    sign = np.sign(d)
    nz = np.nonzero(sign)[0]
    if len(nz) == 0:
        return None
    # Flat stretches inherit the direction of the preceding motion.
    sign[: nz[0]] = sign[nz[0]]
    for k in range(1, len(sign)):
        if sign[k] == 0:
            sign[k] = sign[k - 1]

    # Maximal direction runs over diff indices [a, b): points seg[a..b].
    runs = []
    a = 0
    for k in range(1, len(sign) + 1):
        if k == len(sign) or sign[k] != sign[a]:
            runs.append([a, k, int(sign[a])])
            a = k

    # Persistence merge: runs shorter than TURN_MIN_RUN and runs that
    # continue the previous direction are absorbed into the run before
    # them. Kept runs therefore alternate direction.
    kept = []
    for a, b, direction in runs:
        if (b - a) < TURN_MIN_RUN or (kept and kept[-1][2] == direction):
            if kept:
                kept[-1][1] = b
            continue
        kept.append([a, b, direction])
    if not kept:
        return None

    # The terminal rise: the kept rising run containing the global
    # maximum. Its start is the last valley — the second vertical
    # tangent of the S. If that run is the first kept run, the trace
    # never fell on its way up: no fold.
    gmax = int(np.argmax(seg))
    run_idx = None
    for i, (a, b, direction) in enumerate(kept):
        if direction == 1 and a <= gmax <= b:
            run_idx = i
    if run_idx is None or run_idx == 0:
        return None
    return i0 + kept[run_idx][0]


def extract_oer_branch(eta, J):
    """Isolate the single-valued OER branch via the vertical line test.

    A CV in η vs log₁₀(J) coordinates is parametric: the pre-OER oxide
    wave makes J rise, fall, then rise again, tracing an S that fails
    the vertical line test. ``find_second_turning_point`` locates the
    second vertical tangent (the upper-left fold), and the OER branch
    is everything after it; the Tafel region begins almost immediately
    afterward. When no second turning point exists the curve is
    single-valued and is used whole.

    The non-OER background (valley of smoothed |J| below ``ETA_MIN``,
    the local minimum between the oxide wave and onset) is subtracted,
    points must satisfy η > ``ETA_MIN`` and corrected J > ``SNR_MIN`` ×
    background, and an iterated monotone trim guarantees the returned
    branch is single-valued.

    Returns
    -------
    dict with ``x`` (log₁₀ J_OER), ``y`` (η), ``baseline``,
    ``multivalued``, ``turn_index`` (scan index of the second turning
    point, or None), and ``turn_xy`` (its η–log₁₀ J coordinates for
    diagnostics, or None); or ``None`` when no valid branch exists.
    """
    eta = np.asarray(eta, dtype=float)
    J = np.asarray(J, dtype=float)
    if len(J) < 20:
        return None

    Js = pd.Series(J).rolling(7, center=True, min_periods=1).median().values

    # ── Second turning point → branch start ──────────────────
    turn2 = find_second_turning_point(Js)
    multivalued = turn2 is not None
    start0 = turn2 + 1 if multivalued else 0

    turn_xy = None
    if multivalued and Js[turn2] > 0:
        turn_xy = (float(np.log10(Js[turn2])), float(eta[turn2]))

    # ── Background: valley between oxide wave and OER onset ──
    abs_smooth = (
        pd.Series(np.abs(J)).rolling(7, center=True, min_periods=1).median().values
    )
    pre = abs_smooth[eta < ETA_MIN]
    if len(pre) < 5:
        return None
    baseline = pre.min()

    # ── Subtraction + validity gates on the branch ───────────
    eta_b, J_b = eta[start0:], J[start0:]
    J_corr = J_b - baseline
    Jc_smooth = (
        pd.Series(J_corr).rolling(5, center=True, min_periods=1).median().values
    )
    ok = (eta_b > ETA_MIN) & (Jc_smooth > SNR_MIN * baseline) & (J_corr > 0)
    if not ok.any():
        return None
    fails = np.where(~ok)[0]
    start = (
        fails[-1] + 1
        if len(fails) and fails[-1] >= ok.argmax()
        else ok.argmax()
    )
    if start >= len(eta_b) - 10:
        return None
    J_f, eta_f = J_corr[start:], eta_b[start:]

    # ── Monotone trim, iterated to a fixed point ─────────────
    # (Re-smoothing a trimmed array shifts edge medians and can expose
    # a new small decrease, so the trim repeats until stable.)
    while True:
        Jf_smooth = (
            pd.Series(J_f).rolling(7, center=True, min_periods=1).median().values
        )
        drops = np.where(np.diff(Jf_smooth) < 0)[0]
        if not len(drops):
            break
        J_f, eta_f = J_f[drops[-1] + 1 :], eta_f[drops[-1] + 1 :]
        if len(J_f) < 10:
            return None
    if len(J_f) < 10:
        return None

    return {
        'x': np.log10(J_f),
        'y': eta_f,
        'baseline': baseline,
        'multivalued': multivalued,
        'turn_index': turn2,
        'turn_xy': turn_xy,
    }


def tafel_sliding_window(
    log_j,
    eta,
    anchored,
    window_decades=WINDOW_DECADES,
    density_min=DENSITY_MIN,
    plateau_tol=PLATEAU_TOL,
    local_r2_thresholds=LOCAL_R2_THRESHOLDS,
    slope_max=SLOPE_MAX,
    min_fit_decades=MIN_FIT_DECADES,
):
    """Locate the Tafel region on an extracted OER branch.

    Local slope dη/d(log₁₀ J) and local linearity are computed in
    sliding windows; each window's slope is attributed to its center.
    Candidate gates, each a necessary Tafel condition: positive slope;
    local R² above threshold (kinetic control shows as local
    linearity); ≥ ``density_min`` points/decade (fit stability); slope
    ≤ ``slope_max`` (Butler–Volmer plausibility, b ≲ 240 mV/dec at
    25 °C for α ≳ 0.25).

    Candidate windows form contiguous runs, each holding one slope
    plateau. When ``anchored`` is True (a second turning point was
    found), runs are taken first to last, so the accepted region
    begins as close to the turning point as the data allows — the
    Tafel region starts almost immediately after the second vertical
    tangent. When ``anchored`` is False (single-valued curve), runs
    are taken last to first: the Tafel region is the last plateau
    before the terminal conditions become unsatisfactory at the right
    edge. Within the chosen run the plateau is the minimum local slope
    expanded by ``plateau_tol``, and the fit is truncated at the
    transport bend — the first later window whose (median-smoothed)
    slope rises above the plateau by more than the tolerance. Samples
    with no acceptable region return ``None``; the target is never
    fabricated. Accepted slopes just below ``slope_max`` may be
    censored by the ceiling; treat with care.

    Returns
    -------
    dict with ``slope`` (V/decade), ``intercept``, ``r2``, ``decades``,
    ``n_points``, ``x_bound`` (transport bend, +inf if none), plus
    sorted ``x``, ``y`` arrays and a boolean ``fit_mask``; or ``None``.
    """
    order = np.argsort(log_j)
    x = np.asarray(log_j, dtype=float)[order]
    y = np.asarray(eta, dtype=float)[order]
    n = len(x)

    min_pts = int(np.ceil(window_decades * density_min))

    starts, local_slopes, local_r2, window_ends = [], [], [], []
    for i in range(n):
        stop = np.searchsorted(x, x[i] + window_decades, side='right')
        if stop - i < min_pts:
            continue
        m, b = np.polyfit(x[i:stop], y[i:stop], 1)
        yy = y[i:stop]
        ss_tot = float(((yy - yy.mean()) ** 2).sum())
        r2w = (
            1.0 - float(((yy - (m * x[i:stop] + b)) ** 2).sum()) / ss_tot
            if ss_tot > 0
            else 0.0
        )
        starts.append(i)
        local_slopes.append(m)
        local_r2.append(r2w)
        window_ends.append(x[stop - 1])

    if not local_slopes:
        return None

    starts = np.array(starts)
    local_slopes = np.array(local_slopes)
    local_r2 = np.array(local_r2)
    window_ends = np.array(window_ends)
    centers = (x[starts] + window_ends) / 2

    smoothed = local_slopes.copy()
    if len(local_slopes) >= 3:
        smoothed[1:-1] = np.median(
            np.column_stack(
                [local_slopes[:-2], local_slopes[1:-1], local_slopes[2:]]
            ),
            axis=1,
        )

    # ── Candidate gates ──────────────────────────────────────
    base = (smoothed > 0) & (smoothed <= slope_max)
    candidate = None
    for r2_min in local_r2_thresholds:
        mask = base & (local_r2 >= r2_min)
        if mask.any():
            candidate = mask
            break
    if candidate is None:
        return None

    # ── Contiguous candidate runs, ordered by anchoring rule ─
    cand_idx = np.where(candidate)[0]
    splits = np.where(np.diff(cand_idx) > 1)[0] + 1
    runs = np.split(cand_idx, splits)
    if not anchored:
        runs = runs[::-1]

    for run in runs:
        # Plateau: minimum local slope in the run, expanded within
        # tolerance but never beyond the run.
        plateau_pos = run[int(smoothed[run].argmin())]
        plateau = smoothed[plateau_pos]
        ok = candidate & (smoothed <= plateau * (1 + plateau_tol))
        lo = plateau_pos
        while lo > run[0] and ok[lo - 1]:
            lo -= 1
        hi = plateau_pos
        while hi < run[-1] and ok[hi + 1]:
            hi += 1

        # Transport bend: first later window whose smoothed slope
        # rises above the plateau by more than the tolerance. A branch
        # whose slope never turns up has no detectable transport
        # contamination and may be used to its end.
        upturn = (np.arange(len(smoothed)) > hi) & (
            smoothed > plateau * (1 + plateau_tol)
        )
        x_bound = centers[np.where(upturn)[0][0]] if upturn.any() else np.inf

        # ── Global fit, truncated at the transport bend ──────
        i0 = starts[lo]
        i1 = np.searchsorted(
            x, min(x[starts[hi]] + window_decades, x_bound), side='right'
        )
        xf, yf = x[i0:i1], y[i0:i1]

        if len(xf) < min_pts:
            continue
        decades = xf[-1] - xf[0]
        if decades < min_fit_decades:
            continue

        slope, intercept = np.polyfit(xf, yf, 1)
        resid = yf - (slope * xf + intercept)
        ss_tot = float(((yf - yf.mean()) ** 2).sum())
        r2 = 1.0 - float((resid**2).sum()) / ss_tot if ss_tot > 0 else np.nan

        fit_mask = np.zeros(n, dtype=bool)
        fit_mask[i0:i1] = True

        return {
            'slope': slope,
            'intercept': intercept,
            'r2': r2,
            'decades': decades,
            'n_points': int(i1 - i0),
            'x_bound': x_bound,
            'x': x,
            'y': y,
            'fit_mask': fit_mask,
        }

    return None


In [3]:
# ── Load composition + iR-drop lookup table ──────────────────
catalyst_df = load_catalyst_summary('data/PtPdAuIr_summary.csv')

SCAN_LEN = 949  # number of data rows per half-cycle

os.makedirs(DJDV_DIR, exist_ok=True)
os.makedirs(IMAGE_DIR, exist_ok=True)
os.makedirs(TAFEL_DIR, exist_ok=True)
os.makedirs(TAFEL_DIAG_DIR, exist_ok=True)

cv_files = list_data_files(CV_DIR)

with open('data/composition_Tafel.csv', 'w') as tafel_file, open(
    'data/composition_eta.csv', 'w'
) as eta_file:

    tafel_file.write(
        'Pt,Pd,Au,Ir,Tafel slope,Tafel R2,Tafel decades,Tafel n,J baseline\n'
    )
    eta_file.write('Pt,Pd,Au,Ir,Eta\n')

    for filename in cv_files:
        if filename == '.DS_Store':
            continue

        # ── Match filename → composition row ─────────────────
        cat_row = match_composition(catalyst_df, filename)
        Pt, Pd, Au, Ir = composition_values(cat_row)

        # ── Extract 5th forward scan (LSV) ───────────────────
        cv_df = pd.read_csv(CV_DIR + filename)
        scan_start = 0
        with open(CV_DIR + filename, 'r') as f:
            for line_num, line in enumerate(f, start=1):
                if line[:6] == 'CURVE5':
                    scan_start = line_num + 1  # +1 for header row

        scan_end = scan_start + SCAN_LEN
        lsv = cv_df.iloc[scan_start:scan_end, 3:5].copy()
        lsv.columns = ['V', 'I']
        lsv['V'] = pd.to_numeric(lsv['V'])
        lsv['I'] = pd.to_numeric(lsv['I'])

        dtp = lsv.copy(deep=True)  # preserve raw data for dJ/dV branch
        lsv_raw = lsv.copy(deep=True)  # raw scan for Tafel branch extraction

        # ── iR correction + convert to η and current density ──
        R = cat_row['iR drop (Ω)'].values[0]
        lsv['V'] = lsv['V'] - lsv['I'] * R  # iR-corrected voltage
        lsv['V'] = lsv['V'] - Vo  # overpotential η = V_corr − V_eq
        lsv['I'] = lsv['I'] / AREA  # A → mA/cm² (still in A here)
        lsv = lsv[lsv['I'] > 0].copy()

        # ── η at 10 mA/cm² ───────────────────────────────────
        idx = (lsv['I'] - 0.01).abs().idxmin()
        eta10 = lsv.loc[idx, 'V']
        eta_file.write(f'{Pt},{Pd},{Au},{Ir},{eta10}\n')

        # ── Differential Tafel (dJ/dV) ───────────────────────
        # The dJ/dV curves themselves are plotted (and saved to
        # images/djdVgraphs/) by dJdVpreprocess.ipynb; here we only
        # compute and store the numerical curves.
        dtp = dtp.groupby('V', as_index=False).mean()
        low_mask = dtp['V'] < 1.0
        dtp.loc[low_mask, 'I'] = (
            dtp.loc[low_mask, 'I'].rolling(window=5, center=True).mean()
        )
        V_arr = dtp['V'].values
        J_arr = (dtp['I'] / AREA).values

        dJdV = np.gradient(J_arr, V_arr)

        out_path = os.path.join(DJDV_DIR, filename)
        np.savetxt(
            out_path,
            np.column_stack((V_arr, dJdV)),
            delimiter=',',
            header='V,dJdV',
            comments='',
        )

        # ── Tafel slope: vertical line test + plateau detection ──
        # The branch extractor traces the scan left to right and takes
        # everything after the second vertical tangent (the upper-left
        # fold of the S); the Tafel fit is anchored to begin almost
        # immediately after that turning point. When no second vertical
        # tangent exists, the curve is single-valued and the Tafel
        # region is the last plateau before the terminal conditions
        # become unsatisfactory at the right edge. The background
        # (valley current between oxide wave and onset) is subtracted,
        # and the necessary Tafel conditions (η floor,
        # signal-to-background, transport bend, local linearity,
        # Butler–Volmer plausibility) gate the fit region.
        eta_arr = lsv_raw['V'].values - lsv_raw['I'].values * R - Vo
        J_arr_raw = lsv_raw['I'].values / AREA
        branch = extract_oer_branch(eta_arr, J_arr_raw)
        fit = (
            tafel_sliding_window(
                branch['x'], branch['y'], anchored=branch['multivalued']
            )
            if branch is not None
            else None
        )

        if fit is not None:
            tafel_slope = fit['slope']
            tafel_file.write(
                f'{Pt},{Pd},{Au},{Ir},{tafel_slope},'
                f"{fit['r2']},{fit['decades']},{fit['n_points']},"
                f"{branch['baseline']}\n"
            )
        else:
            tafel_slope = np.nan
            bl = branch['baseline'] if branch is not None else np.nan
            tafel_file.write(f'{Pt},{Pd},{Au},{Ir},nan,nan,nan,0,{bl}\n')

        # ── Save plots to disk (never displayed) ─────────────
        # Journal-style plot (images/Tafel/): the Tafel region only,
        # as it would appear in a figure — fit-region data + fit line.
        # Diagnostic plot (images/TafelDiagnostics/): the full curve
        # with the detected region highlighted, for auditing the
        # extraction.
        sample_id = cat_row.iloc[0, 0]
        plot_name = os.path.splitext(filename)[0]
        composition_label = f'Pt({Pt}), Pd({Pd}), Au({Au}), Ir({Ir})'

        if fit is not None:
            xm = fit['x'][fit['fit_mask']]
            ym = fit['y'][fit['fit_mask']]
            xl = np.array([xm[0], xm[-1]])
            fit_label = (
                f"{fit['slope'] * 1000:.0f} mV dec$^{{-1}}$"
            )

            # Journal-style plot: fit region only.
            fig, ax = plt.subplots(figsize=(6, 4.5))
            ax.scatter(xm, ym, s=18, color='black', zorder=2)
            ax.plot(
                xl,
                fit['slope'] * xl + fit['intercept'],
                color='crimson',
                linewidth=1.8,
                zorder=3,
                label=fit_label,
            )
            ax.set_xlabel(r'log$_{10}$ $J_{\mathrm{OER}}$  (A cm$^{-2}$)')
            ax.set_ylabel(r'$\eta$ (V)')
            ax.set_title(composition_label, fontsize=11)
            ax.legend(loc='lower right', frameon=False)
            plt.tight_layout()
            plt.savefig(
                os.path.join(TAFEL_DIR, f'Tafel_{plot_name}.png'), dpi=300
            )
            plt.close('all')

        # Diagnostic plot: full curve + detected region + gates outcome.
        fig, ax = plt.subplots(figsize=(8, 5))
        positive = J_arr_raw > 0
        ax.scatter(
            np.log10(J_arr_raw[positive]),
            eta_arr[positive],
            s=12,
            color='0.75',
            label='All data (raw J)',
        )
        if branch is not None:
            ax.scatter(
                branch['x'],
                branch['y'],
                s=12,
                color='goldenrod',
                label='OER branch (J − background)',
            )
            if branch['turn_xy'] is not None:
                ax.axvline(
                    branch['turn_xy'][0],
                    color='darkorchid',
                    linestyle='-',
                    linewidth=1,
                    alpha=0.6,
                )
                ax.scatter(
                    [branch['turn_xy'][0]],
                    [branch['turn_xy'][1]],
                    marker='*',
                    s=180,
                    color='darkorchid',
                    zorder=5,
                    label='2nd turning point (vertical tangent)',
                )
            if fit is not None and np.isfinite(fit['x_bound']):
                ax.axvline(
                    fit['x_bound'],
                    color='gray',
                    linestyle='--',
                    linewidth=1,
                    label='Fit end (slope upturn)',
                )
        ax.axhline(0.05, color='gray', linestyle=':', linewidth=1)
        if fit is not None:
            ax.scatter(xm, ym, s=14, color='steelblue', label='Fit region')
            ax.plot(
                xl,
                fit['slope'] * xl + fit['intercept'],
                color='crimson',
                linewidth=2,
                label=(
                    f"{fit['slope'] * 1000:.0f} mV/dec, "
                    f"R²={fit['r2']:.3f}, "
                    f"{fit['decades']:.2f} dec"
                ),
            )
        else:
            ax.annotate(
                'No acceptable Tafel region\n(slope = NaN)',
                xy=(0.05, 0.9),
                xycoords='axes fraction',
                color='crimson',
            )
        ax.set_xlabel(r'log$_{10}$ J  (A/cm²)')
        ax.set_ylabel('η [V]')
        ax.set_title(f'Tafel Diagnostics — {composition_label}')
        ax.legend(loc='best', fontsize=9)
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig(
            os.path.join(TAFEL_DIAG_DIR, f'TafelDiag_{plot_name}.png'),
            dpi=300,
        )
        plt.close('all')  # Free figure memory each iteration.

        slope_txt = (
            f'{tafel_slope * 1000:.1f} mV/dec'
            if np.isfinite(tafel_slope)
            else 'no acceptable fit (NaN)'
        )
        print(f'sample {sample_id}: η₁₀ = {eta10:.4f} V, Tafel slope = {slope_txt}')


sample 174: η₁₀ = 0.1542 V, Tafel slope = 99.3 mV/dec
sample 164: η₁₀ = 0.3191 V, Tafel slope = 155.9 mV/dec
sample 111: η₁₀ = 0.5205 V, Tafel slope = no acceptable fit (NaN)
sample 121: η₁₀ = 0.5199 V, Tafel slope = 170.5 mV/dec
sample 99: η₁₀ = 0.4730 V, Tafel slope = 154.6 mV/dec
sample 60: η₁₀ = 0.5390 V, Tafel slope = 238.4 mV/dec
sample 182: η₁₀ = 0.1439 V, Tafel slope = 88.2 mV/dec
sample 155: η₁₀ = 0.1645 V, Tafel slope = 101.1 mV/dec
sample 140: η₁₀ = 0.3631 V, Tafel slope = 130.2 mV/dec
sample 24: η₁₀ = 0.4139 V, Tafel slope = 87.7 mV/dec
sample 34: η₁₀ = 0.5138 V, Tafel slope = no acceptable fit (NaN)
sample 12: η₁₀ = 0.5164 V, Tafel slope = no acceptable fit (NaN)
sample 48: η₁₀ = 0.3257 V, Tafel slope = 173.0 mV/dec
sample 120: η₁₀ = 0.5490 V, Tafel slope = 244.9 mV/dec
sample 92: η₁₀ = 0.5293 V, Tafel slope = 178.0 mV/dec
sample 78: η₁₀ = 0.2877 V, Tafel slope = 128.1 mV/dec
sample 68: η₁₀ = 0.5728 V, Tafel slope = 126.0 mV/dec
sample 4: η₁₀ = 0.1837 V, Tafel slope = 126.